# LC-MS Metabolomics Data Preprocessing

**Tier 3 — Applied Bioinformatics | Module 36 · Notebook 1**

*Prerequisites: Module 06 (Statistics for Bioinformatics), Module 18 (Proteomics & Structural Methods)*

---

**By the end of this notebook you will be able to:**
1. Describe LC-MS and GC-MS metabolomics experimental workflows
2. Process raw data with XCMS: peak picking, alignment, gap filling
3. Understand m/z, retention time, and adduct formation in mass spectra
4. Apply normalization strategies (PQN, IS-based, LOESS) for batch correction
5. Perform exploratory analysis: PCA, heatmaps, missing value imputation


**Key resources:**
- [XCMS documentation](https://www.bioconductor.org/packages/release/bioc/html/xcms.html)
- [MetaboAnalyst 6.0](https://www.metaboanalyst.ca/)
- [HMDB (Human Metabolome Database)](https://hmdb.ca/)
- [Galaxy Metabolomics training](https://training.galaxyproject.org/training-material/topics/metabolomics/)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Module Overview](../README.md) | [Next: Metabolite Identification →](./02_metabolite_identification.ipynb)

---

## 1. Metabolomics Overview

> Metabolome definition: small molecules < 1500 Da. Untargeted vs targeted vs semi-targeted approaches. LC-MS (liquid chromatography) vs GC-MS (volatile compounds, requires derivatization) vs NMR. Sample types: plasma, urine, tissue, cells.

## 2. LC-MS Data Acquisition

> Chromatographic separation (RPLC, HILIC). ESI ionization: positive vs negative mode. Data-dependent acquisition (DDA) vs data-independent (SWATH/DIA). mzML open format.

## 3. XCMS Peak Picking and Alignment

> centWave algorithm for peak detection. Retention time alignment with obiwarp. Correspondence: match peaks across samples. Gap filling for missing values. Feature table: rows = features (m/z, RT), columns = samples.

In [ ]:
# Example: XCMS processing in R (via subprocess or rpy2)
# # Alternative: use ms-data-core-api or pyOpenMS in Python
# from pyopenms import *
# exp = MSExperiment()
# MzMLFile().load('sample.mzML', exp)

## 4. Normalization and Quality Control

> Probabilistic Quotient Normalization (PQN). Internal standard normalization. QC pooled sample injection for LOESS correction. Coefficient of variation (CV) threshold for feature filtering (< 30% in QC samples).

## 5. Exploratory Analysis

> PCA score plot (colored by condition, batch). OPLS-DA for supervised class separation. Coefficient of variation distribution. Missing value pattern (MCAR vs MNAR) and imputation strategies (kNN, min/2).